# 02: Customer Feature Engineering, RFM Analysis & K-Means Segmentation
This notebook walks through calculating RFM and multi-dimensional engagement metrics, transforming skewed distributions with log1p and robust scaling, evaluating optimal K, and generating deterministic descriptive cluster profiles.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

# Robust root resolver whether run from notebooks/ or workspace root
_curr = Path('.').resolve()
PROJECT_ROOT = _curr if (_curr / 'src').exists() else (_curr.parent if (_curr.parent / 'src').exists() else (_curr.parents[1] if len(_curr.parents) > 1 and (_curr.parents[1] / 'src').exists() else _curr))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Switch working directory to project root so all relative paths ('data/...', 'artifacts/...') resolve seamlessly
os.chdir(PROJECT_ROOT)

import matplotlib.pyplot as plt
from src.features.customer_features import CustomerFeatureEngineer
from src.segmentation.clustering import CustomerClustering
from src.segmentation.profiling import SegmentProfiler
from src.segmentation.visualization import SegmentVisualizer


## 1. Extract Customer Features & RFM Metrics on Train Split

In [ ]:
train_events_df = pd.read_parquet('data/processed/train_events.parquet')
cust_fe = CustomerFeatureEngineer(scaler_type='robust')
features_df = cust_fe.extract_features(train_events_df)
transformed_df, scaled_matrix = cust_fe.transform_and_scale(features_df, fit=True)
features_df.head()


## 2. Evaluate Optimal K via Inertia & Silhouette Diagnostics

In [ ]:
clustering = CustomerClustering(n_clusters=4, random_state=42)
diag = clustering.find_optimal_k(scaled_matrix, k_range=[2, 3, 4, 5, 6, 7, 8])
cluster_labels = clustering.fit_predict(scaled_matrix)
features_df['cluster'] = [int(c) for c in cluster_labels]


## 3. Automated Deterministic Segment Profiling (No Hardcoding)

In [ ]:
profiler = SegmentProfiler()
profiles_df, labels_map = profiler.profile_segments(features_df)
print("Discovered Customer Segments:")
print(profiles_df[['cluster_id', 'label', 'customer_count', 'percentage_of_customers', 'avg_monetary_value', 'avg_frequency', 'avg_recency_days']])


## 4. 2D PCA Dimensionality Reduction Projection

In [ ]:
viz = SegmentVisualizer(n_components=2)
viz.plot_pca_clusters(scaled_matrix, cluster_labels, cluster_names=labels_map, output_image_path='outputs/figures/customer_segments_pca.png')
import matplotlib.image as mpimg
img = mpimg.imread('outputs/figures/customer_segments_pca.png')
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.show()
